# PetMind — EfficientNet-B0 감정 분류 학습 (Kaggle)

### 실행 전 필수 설정
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On**
3. **Add-ons → Secrets → ROBOFLOW_API_KEY** 등록 확인
4. 위에서 아래로 셀 순서대로 실행

In [ ]:
# 1. GPU 확인
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. 패키지 설치
!pip install -q roboflow

In [ ]:
# 3. API 키 로드
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
ROBOFLOW_API_KEY = secrets.get_secret('ROBOFLOW_API_KEY')
print('API 키 로드 완료')

In [ ]:
# 4. 감정 데이터셋 다운로드 및 크롭
import os
import shutil
import yaml
from pathlib import Path
from PIL import Image
from roboflow import Roboflow

DATA_DIR = Path('/kaggle/working/emotion_data')

LABEL_MAP = {
    'happy':   'happy',
    'relaxed': 'neutral',
    'sad':     'sad',
    'angry':   'angry',
}
OUR_LABELS = ['happy', 'sad', 'angry', 'neutral']

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('dog-emotion-zaveh').project('dog-emotion-ovhny')
version = project.version(2)
dataset = version.download('yolov8', location='/kaggle/working/emotion_raw')

with open(Path(dataset.location) / 'data.yaml') as f:
    cfg = yaml.safe_load(f)
class_names = cfg.get('names', [])
print('원본 클래스:', class_names)

def crop_and_save(img_path, txt_path, split):
    if not txt_path.exists():
        return
    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    for i, line in enumerate(txt_path.read_text().splitlines()):
        parts = line.strip().split()
        if not parts:
            continue
        cls_idx = int(parts[0])
        if cls_idx >= len(class_names):
            continue
        mapped = LABEL_MAP.get(class_names[cls_idx])
        if not mapped:
            continue
        cx, cy, bw, bh = map(float, parts[1:5])
        x1 = max(0, int((cx - bw/2) * w))
        y1 = max(0, int((cy - bh/2) * h))
        x2 = min(w, int((cx + bw/2) * w))
        y2 = min(h, int((cy + bh/2) * h))
        if x2 <= x1 or y2 <= y1:
            continue
        dst_dir = DATA_DIR / split / mapped
        dst_dir.mkdir(parents=True, exist_ok=True)
        img.crop((x1, y1, x2, y2)).save(dst_dir / f'{img_path.stem}_{i}.jpg')

split_map = {'train': 'train', 'valid': 'val', 'test': 'test'}
for src_split, dst_split in split_map.items():
    img_dir = Path(dataset.location) / src_split / 'images'
    lbl_dir = Path(dataset.location) / src_split / 'labels'
    if not img_dir.exists():
        continue
    for img_path in img_dir.iterdir():
        crop_and_save(img_path, lbl_dir / (img_path.stem + '.txt'), dst_split)

print('\n=== 데이터 준비 완료 ===')
for split in ['train', 'val', 'test']:
    split_dir = DATA_DIR / split
    if not split_dir.exists():
        continue
    row = [f"{l}:{len(list((split_dir/l).glob('*.jpg'))) if (split_dir/l).exists() else 0}" for l in OUR_LABELS]
    print(f'  {split}: {"  ".join(row)}')

In [ ]:
# 5. 모델 정의 (EfficientNet-B0)
import torch
import torch.nn as nn
from torchvision import models

LABELS = ['happy', 'sad', 'angry', 'neutral']

class EmotionClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)

print('모델 정의 완료')

In [ ]:
# 6. 학습
# ⏱️ 예상 시간: T4 기준 약 30~40분 (50 에폭)
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from pathlib import Path

DATA_DIR = Path('/kaggle/working/emotion_data')
WEIGHTS_DIR = Path('/kaggle/working/emotion_weights')
WEIGHTS_DIR.mkdir(exist_ok=True)

TRAIN_TRANSFORMS = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
VAL_TRANSFORMS = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(DATA_DIR / 'train', transform=TRAIN_TRANSFORMS)
val_ds   = datasets.ImageFolder(DATA_DIR / 'val',   transform=VAL_TRANSFORMS)
print('클래스 인덱스:', train_ds.class_to_idx)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=4)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=4)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EmotionClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

best_acc = 0.0
for epoch in range(1, 51):
    model.train()
    for images, labels in train_dl:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in val_dl:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(1) == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    print(f'Epoch {epoch:3d}/50 — val acc: {acc:.4f}')

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), WEIGHTS_DIR / 'best.pt')
        print(f'  → best 저장 (acc={best_acc:.4f})')

print(f'\n학습 완료! Best val accuracy: {best_acc:.4f}')

In [ ]:
# 7. 결과 확인
from pathlib import Path

best_pt = Path('/kaggle/working/emotion_weights/best.pt')
if best_pt.exists():
    size_mb = best_pt.stat().st_size / 1024 / 1024
    print(f'best.pt: {size_mb:.1f} MB')
    print('오른쪽 사이드바 → Output → emotion_weights → best.pt 다운로드')
else:
    print('best.pt 없음')